# Huấn luyện hệ gợi ý tin tức trên MIND-large (Kaggle)

**Cài đặt notebook (cột bên phải):**
1. *Accelerator*: **GPU T4 x2** hoặc **GPU P100**. *Internet*: **On**. *Persistence*: **Files only** (giữ `/kaggle/working` giữa các phiên).
2. *Add-ons → Secrets*: thêm secret tên **`HF_TOKEN`** là token Read của HuggingFace (tài khoản đã đồng ý điều khoản tại https://huggingface.co/datasets/yjw1029/MIND).
3. Mã nguồn: sửa `REPO_URL` bên dưới thành link GitHub của repo, **hoặc** upload thư mục repo (dạng zip) thành Kaggle Dataset rồi *Add Input*.

Mỗi phiên Kaggle tối đa 12 giờ. Các lệnh huấn luyện tự lưu `last.pt` trước khi hết giờ (`train.time_limit_hours`), nên chỉ cần **chạy lại toàn bộ notebook**: bước nào đã xong sẽ được bỏ qua, run đang dở được chạy tiếp.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/<ten-ban>/news-recommender.git"   # <- sửa
WORK = Path("/kaggle/working/news-recommender")

if not WORK.exists():
    code_inputs = [p for p in Path("/kaggle/input").glob("*/*") if (p / "newsrec").exists()]
    if code_inputs:
        subprocess.run(["cp", "-r", str(code_inputs[0]), str(WORK)], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(WORK)], check=True)
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44,<5.0", "huggingface_hub>=0.24", "pyyaml"], check=True)

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Tải và tiền xử lý MIND-large (chỉ chạy lần đầu)

In [ ]:
if not Path("data/processed/mind_large/info.json").exists():
    !python scripts/download_mind.py --size large --glove
    !python -m newsrec.preprocess --raw data/raw --size large --glove data/raw/glove/glove.840B.300d.txt
    !rm -rf data/raw   # giải phóng chỗ trống, chỉ giữ dữ liệu đã tiền xử lý
!cat data/processed/mind_large/info.json | head -40

## 2. Baseline không cần GPU

In [ ]:
!python -m newsrec.baselines --data data/processed/mind_large

## 3. Cache PLM (chạy các tầng đóng băng của MiniLM một lần cho mọi tin)

In [ ]:
!python -m newsrec.plm_cache --data data/processed/mind_large --device cuda --batch-size 512

## 4. Thực nghiệm chính: NRMS tái lập và mô hình đề xuất, 3 seed

`model.plm.cache_device=cuda` đặt cache lên GPU cho nhanh (cần khoảng 6 GB VRAM). Nếu báo hết bộ nhớ GPU, đổi thành `cpu`.

In [ ]:
COMMON = "train.num_workers=4 model.plm.cache_device=cuda"
!python -m newsrec.experiments main --seeds 42 43 44 --set {COMMON}

## 5. Ablation (1 seed, 1 epoch trên 1 triệu mẫu để vừa quota GPU)

Mọi ablation, kể cả dòng tham chiếu `abl_reference`, dùng cùng thiết lập rút gọn nên so sánh được với nhau.

In [ ]:
!python -m newsrec.experiments ablation --seeds 42 --set {COMMON} train.epochs=1 train.max_train_samples=1000000
!python -m newsrec.experiments baselines_extra --seeds 42 --set {COMMON}

## 6. Ensemble, kiểm định thống kê và bảng kết quả

In [ ]:
!python -m newsrec.analysis ensemble runs/full_seed42 runs/full_seed43 runs/full_seed44 --name ens_full
!python -m newsrec.analysis bootstrap runs/full_seed42 runs/nrms_glove_seed42
!python -m newsrec.analysis table

## 7. Đóng gói kết quả để tải về máy (chạy demo web)

Tải `results_package.zip` trong tab *Output*, giải nén vào thư mục repo trên máy.

In [ ]:
import zipfile
keep = {"config.yaml", "metrics.json", "best.pt", "news_vecs.npy"}
with zipfile.ZipFile("/kaggle/working/results_package.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in Path("runs").rglob("*"):
        if f.name in keep or (f.name == "dev_scores.npy"):
            z.write(f)
    z.write("results.md")
print(Path("/kaggle/working/results_package.zip").stat().st_size // 2**20, "MB")